# Facet fill runs

How the facet fill (step 3 in `comp_universe_and_facets.ipynb`) got to its current design. It reads what `v0/fill_facets.py` and `v0/check_claude_api.py` saved under `outputs-local/comps/`; it never calls a model and never reads financial fields.

1. Where things stand
2. Prompt versions: what changed
3. Model choice
4. Pages only fill gaps
5. Checks before the first Claude fill
6. Every fill run: tokens, time, cost, fill rate
7. The fill test

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

# repo root = the folder holding data_manifest.json (works from the repo root, v0/ or v0/notebooks/)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_manifest.json").exists())
sys.path.insert(0, str(ROOT / "v0"))
import facets as fc                     # also puts website_scrape/ and src/ on the path
import universe as uv
from fill_facets import company_record, load_profiles
from project_data import OUTPUTS_LOCAL

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

companies, _ = uv.build_universe()      # company information only; financials are not used here
universe = companies[companies["in_universe"]]

# every fill run: one row per business profile, from each folder fill_facets.py wrote
COMPS = OUTPUTS_LOCAL / "comps"
RUN_FOLDERS = [COMPS / "facets", *sorted(p for p in (COMPS / "fill_test").iterdir() if p.is_dir())]
runs = pd.concat([load_profiles(p).assign(run=p.relative_to(COMPS).as_posix()) for p in RUN_FOLDERS],
                 ignore_index=True)
runs = runs[runs["llm_status"] == "ok"].merge(
    companies[["IQ_COMPANY_ID", "target_ciq_ticker", "target_company_name"]], on="IQ_COMPANY_ID", how="left")
GROUP = ["run", "prompt_version", "model"]

# the fill test's companies (section 7)
PICKED = ["PCTY", "AME", "BMRN", "ED", "DHIL", "CRD.B", "CPRT", "BBWI", "AVA", "ROAD", "ADUS", "AMTM"]
RANDOM = universe[~universe["target_ciq_ticker"].isin(PICKED)].sample(18, random_state=0)["target_ciq_ticker"].tolist()
TEST_TICKERS = PICKED + RANDOM + ["IMKT.A"]


def fill_pct(t: pd.DataFrame) -> float | None:
    """Share of facets not unknown, over business profiles."""
    return round(100 * t["facets_filled"].sum() / t["facets_total"].sum(), 1) if len(t) else None


print(f"current prompt version {fc.PROMPT_VERSION}; {runs.groupby(GROUP).ngroups} saved runs in {len(RUN_FOLDERS)} folders")

current prompt version 2026-10-25k; 8 saved runs in 5 folders


## 1. Where things stand (2026-10-24)

- **Current design:** prompt version k with Claude Haiku 5.5 through the Anthropic API: one profile per company, filled from what the model knows (name and website), then the saved website pages only for facets left unknown. The model describes only companies it knows; one it doesn't know comes back all unknown and goes to the pages. Splitting a company into its businesses is switched off for now (versions f-j had it; `facets.SPLIT_BUSINESSES`).
- **The full run is done at version k (2026-10-25):** all 533 companies, 6,389 of 6,396 facets filled; 19 companies the model didn't know were filled from their pages; $0.55 for 552 calls (section 6).
- **Fixed before any real run:** at version h the checks (section 5) showed the model inventing a plausible profile from a made-up company's name. Version i stops that, and the checks show it: made-up companies come back unknown, a well-known one is still filled.
- **The fill test ran at version i** (31 companies, $0.041 for both passes, section 7): 29 companies the model knew were filled in the first pass; the two it didn't know were filled from their pages in the second pass (one after its website was rescraped: the first scrape had saved a browser error screen). Every facet of every company is filled. Five companies were split into businesses. Against the earlier Codex answers, the category facets where several apply mostly differ in how many categories are listed, not which.
- **Known limit:** the model's knowledge can be years out of date (American Financial's annuities, sold in 2021, are still in its products). Companies whose pages mention a sale, divestiture or spin-off are flagged for the manual review (13); other out-of-date facts are left to that review.
- **Next:** the manual review, and recipe step 4 (suggest comps).

## 2. Prompt versions: what changed

All versions were made on 2026-10-24; `facets.PROMPT_VERSION` is the current one, and a profile made with an older version is redone by the next run.

| version | what changed | why |
|---|---|---|
| a | The model read the company's website text only, with the name and website masked as "the Company", and was told not to use what it knew. Facets came with a separate tags field. | The method has to work for private companies nobody has heard of. |
| b | Example values for each facet; the categories became the value. | not recorded |
| c | A category plus a short detail; source URLs per facet; company size from the website. | not recorded |
| d | Company size as words only; the revenue band removed. | Keep the facets non-financial. |
| e | **Model knowledge first:** the name and website given, no masking; the website pages optional; each facet's source is model knowledge, website or both. | The website-only design left gaps wherever a site didn't say something, and no answer at all for the 20 companies whose sites gave under 40 words. |
| f | One profile per business for unrelated businesses (B1 the largest, at most four); "unknown" only as a last resort, with a best description and a note instead. | The recipe expects fills close to 100%; a conglomerate has to be matched business by business. |
| g | Shorter answers (one-line JSON, details of about twelve words); no source asked when there are no pages; page lines labeled and cited by label instead of copied quotes. | Codex's usage window allowed about 200 calls, and output is most of what a call costs; copied quotes sometimes failed to match the page. |
| h | Company size is a category with no text (small, mid-sized, large, among the largest); core offering in two or three sentences, lists of up to about five, details of about twenty words; page lines with figures left out of the prompt; pages only fill gaps (two passes); Claude Haiku 5.5. | A free-text size could hold spelled-out figures (section 5); with Haiku, longer answers cost almost nothing; scraped text shouldn't override what the model knows (section 4). |
| i | The model first says whether it knows this specific company (`knows_company`); a name or web address alone isn't knowing it, and it must not guess from the name or from what is typical for similar names. In the first pass, "doesn't know" makes every facet unknown in code, whatever the model wrote, and the company goes to the second pass, where the pages are all it may use. | At version h a made-up company came back with most facets invented from its name (section 5); for private targets the model doesn't know, that would give profiles that look real but aren't. |
| j | "Describe the company as it is now": leave out businesses sold, spun off, closed or held as a minority stake; a spelled-out list of what counts as one business ("when in doubt, it is one business"). Plus a third pass for split companies: each business checked against the saved pages, a business they don't show removed unless its name words are on them. | In the first 100 companies, 17 were split, several by reporting segment or region, and some with businesses sold years ago (Carlisle's foodservice unit, sold 2018). In the full run the check removed businesses correctly (LSB, MSG Entertainment, Trinity, Albemarle) but also two that News Corp still owns, because only its homepage was saved. |
| k | **Current.** One profile per company: the schema has room for one business, which covers all of the company's lines, largest first. Splitting stays in the code, switched off (`SPLIT_BUSINESSES`); the third pass doesn't run. | Decided 2026-10-25 to leave business splitting for later: splits varied between runs and the check needs more work. |

The runs saved for each version (fill % = facets not unknown, over business profiles):

In [2]:
g = runs.groupby(["prompt_version", "run", "model"])
by_version = g.agg(companies=("IQ_COMPANY_ID", "nunique"), profiles=("business_id", "size"))
filled = g[["facets_filled", "facets_total"]].sum()
by_version["fill %"] = (100 * filled["facets_filled"] / filled["facets_total"]).round(1)
by_version

companies  profiles  fill %
prompt_version run                              model                                        
2026-10-24a    facets                           default                 137       137    88.7
2026-10-24e    fill_test/knowledge_only         default                  30        30    99.7
               fill_test/with_pages             default                  12        12    99.3
2026-10-24f    fill_test/knowledge_only         default                  21        21   100.0
2026-10-24i    facets                           claude-haiku-5-5        100       118    97.5
               fill_test/knowledge_only_i_haiku claude-haiku-5-5         31        38   100.0
2026-10-24j    facets                           claude-haiku-5-5        533       606    97.2
2026-10-25k    facets                           claude-haiku-5-5        533       533    99.9

No profiles from versions b-d are saved. `facets/` at version a is the website-only design, with facets that differ from today's.

## 3. Model choice

**Codex CLI (every run up to version g).** `codex exec` under a student's ChatGPT sign-in: nothing billed per call, but every call counts against the account's usage window, and each call carries about 15k tokens of Codex's own instructions and tools on top of a prompt of about 3k. About 200 calls fit in one window on 2026-10-24, short of the 533 companies, and a run with a cheaper Codex model hit the limit on its first calls. The model wasn't recorded, so those runs show "default" (Codex's own choice).

**Claude Haiku 5.5 (from version h, decided 2026-10-24).** The Anthropic API, paid per token from a Console account: $0.10 per million input tokens and $0.50 per million output tokens, with cached input at a tenth of the input price. No usage window, so a run doesn't stop halfway. The answer is held to the schema (`output_config.format`), and the instructions and schema, the same for every company, are cached. About $0.001 per company (section 5).

**Considered and set aside:**
- **Claude Code on a Pro plan:** its limits are shared with claude.ai and not published, and each call carries Claude Code's own large system prompt.
- **Llama 3.1 (8B or 70B) with a search API:** its knowledge ends around 2023; 70B doesn't fit this laptop's 6 GB GPU; search results bring valuation figures and peer lists into the input.
- **Small local models** (such as gpt-oss-20b or Qwen3.5 4B): free, but they know little about mid-size companies; they are good only at reading pages.
- **A larger Claude model** (Sonnet 5.5, about twenty times Haiku's price): only if the fill test shows Haiku's facets are poor.

**Rules whatever the model:** one model for every company in a run (a profile made with another model is redone); reasoning effort left at the model's default (medium on Haiku 5.5); the model recorded in every call's `run.json` and in the profile. `--backend codex` still works.

## 4. Pages only fill gaps

**Decision (2026-10-24):** the scraped website pages are a fallback, never the first source. A scraped description can be thin or one-sided, and given up front it pulls the answer toward the website's wording.

1. **First pass, every company:** the model fills the facets from what it knows, given only the name and website.
2. **Second pass, only if needed:** a company left with an unknown facet gets one more call with the pages the website scrape already saved (no new scraping). Code copies over only the facets that were unknown (`facets.fill_gaps`, matching B1 with B1); nothing filled in the first pass is replaced.
3. **Recorded:** the profile's `gap_fill` says what the second call filled, and `facets.csv` lists it per business (`filled_from_pages`). `--no-gap-fill` runs the first pass only.

What the earlier pages-first run (version e, the 12 hard picks, pages given from the start) showed: the pages didn't raise the fill rate on the same companies,

In [3]:
same12 = runs[runs["target_ciq_ticker"].isin(PICKED) & runs["prompt_version"].eq("2026-10-24e")]
pd.DataFrame([{"run": run, "companies": t["IQ_COMPANY_ID"].nunique(), "fill %": fill_pct(t)}
              for run, t in same12.groupby("run")]).set_index("run")

,companies,fill %
run,,
fill_test/knowledge_only,12,100.0
fill_test/with_pages,12,99.3


but a share of the filled facets then came from the website alone (source website, no model knowledge), so the scraped text, not what the model knew, decided them:

In [4]:
t = same12[same12["with_pages"]]
src = pd.concat([t[f"{f.name}_source"].where(t[f"{f.name}_status"] != "unknown") for f in fc.FACETS]).dropna()
kind = src.map(lambda s: "both" if s.startswith("model knowledge;") else
               "model knowledge" if s == "model knowledge" else "website" if s else "none given")
(kind.value_counts(normalize=True) * 100).round(1).to_frame("% of filled facets, pages from the start")

,"% of filled facets, pages from the start"
both,37.8
model knowledge,35.7
website,26.6


## 5. Checks before the first Claude fill

`python v0/check_claude_api.py` calls the API the way `fill_facets.py` does, with two made-up companies (plausible names and web addresses, nothing saying they're made up) and one well-known real one, Paylocity (about half a cent per run), and saves every answer under `outputs-local/comps/api_checks/<time>/`. Before that script existed, the same checks were run by hand on 2026-10-24 for about $0.0045; those weren't saved. The latest run:

In [5]:
CHECK_RUNS = sorted((COMPS / "api_checks").glob("*/checks.json"))
latest = json.loads(CHECK_RUNS[-1].read_text(encoding="utf-8"))
print(f"latest: {latest['run']} | model {latest['model']} | prompt version {latest['prompt_version']} | "
      f"${latest['total_cost_usd']:.4f}; all {len(CHECK_RUNS)} check runs: "
      f"${sum(json.loads(p.read_text(encoding='utf-8'))['total_cost_usd'] for p in CHECK_RUNS):.4f}")
WHAT = {"digit_rule_one_field": "one-field schema with a no-digits rule; the prompt asks for figures",
        "digit_rule_full_schema": "the full answer schema with the no-digits rule on one text facet",
        "first_call": "current schema, made-up company 1, instructions as the cached system prompt",
        "second_call": "current schema, made-up company 2, cached system prompt",
        "one_message_call": "made-up company 1, the whole prompt as one message, no cache",
        "known_company_call": "a well-known real company (Paylocity), cached system prompt"}
rows = []
for name, c in latest["checks"].items():
    u = c.get("usage") or {}
    if c.get("accepted") is False:
        result = "refused: " + c["error"].split("'message': '")[-1].split("'")[0]
    elif "facets_unknown" in c:
        known = "" if c.get("knows_company") is None else f"knows company: {c['knows_company']}; "
        result = f"accepted; {known}unknown facets {c['facets_unknown']}"
    else:
        result = f"accepted; digit in the answer: {c.get('has_digit')}"
    rows.append({"check": name, "what": WHAT.get(name, ""), "result": result,
                 "input tokens": u.get("input_tokens"), "from cache": u.get("cached_input_tokens"),
                 "written to cache": u.get("cache_write_input_tokens"), "output tokens": u.get("output_tokens"),
                 "seconds": c.get("seconds"), "USD": round(c["cost_usd"], 5)})
pd.DataFrame(rows).set_index("check")

latest: 20261025T045253Z | model claude-haiku-5-5 | prompt version 2026-10-25k | $0.0030; all 6 check runs: $0.0174


,what,result,input tokens,from cache,written to cache,output tokens,seconds,USD
check,,,,,,,,
digit_rule_one_field,one-field schema with a no-digits rule; the prompt asks for figures,accepted; digit in the answer: False,263.0,NaN,NaN,194.0,2.0,0.00012
digit_rule_full_schema,the full answer schema with the no-digits rule on one text facet,refused: Schema is too complex.,NaN,NaN,NaN,NaN,NaN,0.00000
first_call,"current schema, made-up company 1, instructions as the cached system prompt",accepted; knows company: False; unknown facets 12 of 12,5920.0,0.0,5870.0,507.0,5.7,0.00099
second_call,"current schema, made-up company 2, cached system prompt",accepted; knows company: False; unknown facets 12 of 12,5918.0,5870.0,0.0,535.0,3.3,0.00033
one_message_call,"made-up company 1, the whole prompt as one message, no cache",accepted; knows company: False; unknown facets 12 of 12,5920.0,0.0,0.0,486.0,2.8,0.00084
known_company_call,"a well-known real company (Paylocity), cached system prompt",accepted; knows company: True; unknown facets 0 of 12,5903.0,5870.0,0.0,1308.0,6.4,0.00072


The answer to the no-digits check, and the model's own notes on the made-up companies:

In [6]:
print("no-digits rule:", json.loads(latest["checks"]["digit_rule_one_field"]["answer"])["company_size"], "\n")
for name in ("first_call", "second_call", "one_message_call", "known_company_call"):
    print(f"{name}: {latest['checks'][name].get('notes')}")

no-digits rule: one hundred twenty plants, five thousand employees, and two billion dollars in revenue 

first_call: I do not recognize this company from the name and website alone, so all facets are unknown.
second_call: I do not recognize this company from my knowledge, so all facets are unknown.
one_message_call: I do not recognize this company from my knowledge, so all facets are unknown.
known_company_call: 


Every check run so far, by prompt version: how many of the 12 facets came back unknown. At version h the made-up companies were mostly invented from their names; from version i they come back unknown, and the real company is still filled. (The first runs labeled the made-up names "(fictional test company)", which gave the answer away; later runs use plausible names with nothing saying so.)

In [7]:
history = []
for p in CHECK_RUNS:
    d = json.loads(p.read_text(encoding="utf-8"))
    row = {"check run": d["run"], "prompt version": d["prompt_version"], "USD": round(d["total_cost_usd"], 4)}
    for name in ("first_call", "second_call", "one_message_call", "known_company_call"):
        c = d["checks"].get(name)
        row[name + ": unknown"] = c.get("facets_unknown") if c else ""
    history.append(row)
pd.DataFrame(history).set_index("check run")

,prompt version,USD,first_call: unknown,second_call: unknown,one_message_call: unknown,known_company_call: unknown
check run,,,,,,
20261024T234742Z,2026-10-24h,0.0024,1 of 12,1 of 12,,
20261024T234925Z,2026-10-24h,0.0030,3 of 12,2 of 12,3 of 12,
20261025T000440Z,2026-10-24i,0.0030,12 of 12,12 of 12,12 of 12,0 of 12
20261025T000547Z,2026-10-24i,0.0026,12 of 12,12 of 12,12 of 12,0 of 12
20261025T013502Z,2026-10-24j,0.0033,12 of 12,12 of 12,12 of 12,0 of 12
20261025T045253Z,2026-10-25k,0.0030,12 of 12,12 of 12,12 of 12,0 of 12


What the model wrote for made-up company 1 in the latest run (first call):

In [8]:
run_dir = CHECK_RUNS[-1].parent
answer = fc.check_answer(json.loads((run_dir / "first_call" / "response.json").read_text(encoding="utf-8")), {}, {})
b1 = answer["businesses"][0]
pd.DataFrame([{"facet": f"{fc.FACET_ID[k]} {fc.FACET_BY_NAME[k].label}", "status": f["status"], "value": f["value"]}
              for k, f in b1["facets"].items()]).set_index("facet")

,status,value
facet,,
F1 Core offering,unknown,
F2 Primary products,unknown,
F3 Primary services,unknown,
F4 Customer segment,unknown,
F5 End markets,unknown,
F6 Revenue model,unknown,
F7 Value chain position,unknown,
F8 Geography served,unknown,
F9 Company size,unknown,


Prompts checked for figures, every universe company, both passes (`facets.figures_in_input`, the website scrape's financial, size and ticker filters; no model involved):

In [9]:
rows = []
for with_pages in (False, True):
    hits = {}
    for c in universe.itertuples():
        record = company_record(c.IQ_COMPANY_ID, c.target_company_name, c.target_ciq_website, with_pages)
        prompt, lines = fc.build_prompt(record, with_pages)
        found = fc.figures_in_input(prompt, bool(lines))
        if found:
            hits[c.target_ciq_ticker] = found
    rows.append({"pass": "second (with saved pages)" if with_pages else "first (name and website)",
                 "prompts": len(universe), "with a figure": len(hits),
                 "examples": "; ".join(f"{t}: {f[:2]}" for t, f in list(hits.items())[:3])})
pd.DataFrame(rows).set_index("pass")

,prompts,with a figure,examples
pass,,,
first (name and website),533,0,
second (with saved pages),533,0,


**What the checks showed:**
- **A no-digits rule isn't enough.** On a single field the API takes the rule and the model writes no digits, but it spells the figures out instead (the answer above). That is why company size became a category at version h: a fixed list can't hold a figure.
- **The full answer schema can't carry such a rule on Claude:** with it, the API refuses the schema as too complex before the model writes anything (no charge). Without it, the current schema is accepted.
- **Caching works and covers the schema:** after the first call, only about 40 input tokens per company are billed at full price; most of the remaining cost is output, mostly the model's hidden reasoning.
- **No prompt carries a figure:** page lines the filters match are left out (a page title such as "Get Up To $500 In 5 Minutes" got through before version h). This shows what the model is given, not what it remembers from training.
- **The model guessed from the name, until version i.** At version h, for both made-up companies most facets came back filled from what the name suggested (company 1's answer echoed the facet guide's own examples), and the notes said so; putting the instructions in the system prompt or not made no consistent difference. The cause was version f's instruction to give a best description rather than answer unknown. Version i asks the model whether it knows the company and drops everything when it doesn't (decided 2026-10-24); the history table above shows the change, and Paylocity shows that known companies are still filled.

## 6. Every fill run: tokens, time, cost, fill rate

One row per output folder, prompt version and model. Tokens and seconds are medians per first-pass call, from each call's `run.json`. Cost is in US dollars per company for Claude runs (mean, with cache writes at 1.25x the input price); Codex runs aren't billed per call, they use the ChatGPT account's usage window. Second-pass calls aren't in this table.

In [10]:
USD_PER_MTOK = {"claude-haiku-5-5": {"input": 0.10, "cache_write": 0.125, "cache_read": 0.01, "output": 0.50}}

def usd(calls: pd.DataFrame, model: str) -> pd.Series:
    p = USD_PER_MTOK[model]
    cached = calls["cached_input_tokens"].fillna(0)
    written = calls.get("cache_write_input_tokens", pd.Series(0, index=calls.index)).fillna(0)
    return (p["input"] * (calls["input_tokens"] - cached - written) + p["cache_write"] * written
            + p["cache_read"] * cached + p["output"] * calls["output_tokens"]) / 1e6

calls = runs.drop_duplicates(GROUP + ["IQ_COMPANY_ID"])          # one first-pass call per company
summary = calls.groupby(GROUP).agg(
    companies=("IQ_COMPANY_ID", "nunique"), input_tokens=("input_tokens", "median"),
    cached=("cached_input_tokens", "median"), output_tokens=("output_tokens", "median"),
    seconds=("seconds", "median"))
filled = runs.groupby(GROUP)[["facets_filled", "facets_total"]].sum()
summary.insert(1, "fill %", (100 * filled["facets_filled"] / filled["facets_total"]).round(1))
summary["USD per company"] = pd.Series({key: usd(t, key[2]).mean() for key, t in calls.groupby(GROUP)
                                        if key[2] in USD_PER_MTOK}, dtype=float).round(5)
summary

companies  fill %  input_tokens   cached  output_tokens  seconds  USD per company
run                              prompt_version model                                                                                              
facets                           2026-10-24a    default                 137    88.7       18057.0  13696.0         1474.0    24.20              NaN
                                 2026-10-24i    claude-haiku-5-5        100    97.5        5975.0   5942.0         1897.0     9.00          0.00112
                                 2026-10-24j    claude-haiku-5-5        533    97.2        6081.0   6048.0         1886.0     8.80          0.00107
                                 2026-10-25k    claude-haiku-5-5        533    99.9        5903.0   5870.0         1817.0     8.70          0.00097
fill_test/knowledge_only         2026-10-24e    default                  30    99.7       17211.0      0.0          883.0    19.75              NaN
                                 2026-10-24f    default                  21   100.0       17414.0      0.0          917.0    20.00              NaN
fill_test/knowledge_only_i_haiku 2026-10-24i    claude-haiku-5-5         31   100.0        5975.0   5942.0         1784.0     8.70          0.00114
fill_test/with_pages             2026-10-24e    default                  12    99.3       18256.0      0.0         1366.0    24.05              NaN

In [11]:
at_current = summary.reset_index()
at_current = at_current[at_current["prompt_version"] == fc.PROMPT_VERSION]
print(f"runs at the current prompt version ({fc.PROMPT_VERSION}):",
      f"{len(at_current)}" if len(at_current) else "none yet (the fill-test command is in section 7)")
at_current if len(at_current) else None

runs at the current prompt version (2026-10-25k): 1


,run,prompt_version,model,companies,fill %,input_tokens,cached,output_tokens,seconds,USD per company
3,facets,2026-10-25k,claude-haiku-5-5,533,99.9,5903.0,5870.0,1817.0,8.7,0.00097


## 7. The fill test

31 companies, filled the way the full run will be:
- **12 picked to be hard:** seven that had gaps when the model read only the website (Paylocity, AMETEK, BioMarin, Con Ed, Diamond Hill, Crawford, Amentum), three whose sites gave no usable text (Copart, Bath & Body Works, Avista), and two ordinary ones (Construction Partners, Addus).
- **18 drawn at random** from the rest of the universe (`sample(18, random_state=0)`).
- **Ingles Markets,** a likely multi-business company (supermarkets, shopping centers, a milk plant).

Each run goes in its own folder under `fill_test/` and shows up in every table below. The run at the current version:

```
python v0/fill_facets.py --workers 4 --out outputs-local/comps/fill_test/knowledge_only_i_haiku --tickers <the 31, printed below>
```

`with_pages` is the 12 with pages given from the start (version e), the experiment section 4 replaced. Fill rates by group, per business profile:

In [12]:
print("the 31:", " ".join(TEST_TICKERS), "\n")
test = runs[runs["run"].str.startswith("fill_test/")].copy()
test["label"] = (test["run"].str.removeprefix("fill_test/") + " · " + test["prompt_version"].str[-1]
                 + " · " + test["model"])
pd.DataFrame([{"folder · version · model": label, "companies (of 31)": t["IQ_COMPANY_ID"].nunique(),
               "profiles": len(t), "fill %": fill_pct(t),
               "12 picked": fill_pct(t[t["target_ciq_ticker"].isin(PICKED)]),
               "random 18": fill_pct(t[t["target_ciq_ticker"].isin(RANDOM)]),
               "not filled yet": " ".join(sorted(set(TEST_TICKERS) - set(t["target_ciq_ticker"])))
               if t["run"].iloc[0] != "fill_test/with_pages" else ""}
              for label, t in test.groupby("label")]).set_index("folder · version · model")

the 31: PCTY AME BMRN ED DHIL CRD.B CPRT BBWI AVA ROAD ADUS AMTM A950140 DV KORE AZZ COTY BGS ALB HURN HQI NGS MIR JOE REYN GENC CDNL SBC FG PCRX IMKT.A 



,companies (of 31),profiles,fill %,12 picked,random 18,not filled yet
folder · version · model,,,,,,
knowledge_only · e · default,30,30,99.7,100.0,99.5,IMKT.A
knowledge_only · f · default,21,21,100.0,100.0,100.0,HURN IMKT.A JOE KORE MIR NGS PCRX PCTY REYN SBC
knowledge_only_i_haiku · i · claude-haiku-5-5,31,38,100.0,100.0,100.0,
with_pages · e · default,12,12,99.3,99.3,NaN,


Fill rate per facet (% of business profiles not unknown):

In [13]:
pd.DataFrame({label: {f"{fc.FACET_ID[f.name]} {f.label}": 100 * t[f"{f.name}_status"].ne("unknown").mean()
                      for f in fc.FACETS}
              for label, t in test.groupby("label")}).round(1)

,knowledge_only · e · default,knowledge_only · f · default,knowledge_only_i_haiku · i · claude-haiku-5-5,with_pages · e · default
F1 Core offering,100.0,100.0,100.0,100.0
F2 Primary products,96.7,100.0,100.0,91.7
F3 Primary services,100.0,100.0,100.0,100.0
F4 Customer segment,100.0,100.0,100.0,100.0
F5 End markets,100.0,100.0,100.0,100.0
F6 Revenue model,100.0,100.0,100.0,100.0
F7 Value chain position,100.0,100.0,100.0,100.0
F8 Geography served,100.0,100.0,100.0,100.0
F9 Company size,100.0,100.0,100.0,100.0
F10 Product specialization,100.0,100.0,100.0,100.0


Every unknown left, with the model's note on the company:

In [14]:
rows = [{"run": r["label"], "company": r["target_company_name"], "business": r["business_id"],
         "facet": f"{fc.FACET_ID[f.name]} {f.label}", "note": r["notes"]}
        for f in fc.FACETS for _, r in test[test[f"{f.name}_status"] == "unknown"].iterrows()]
pd.DataFrame(rows) if rows else print("no unknowns")

,run,company,business,facet,note
0,knowledge_only · e · default,Huron Consulting Group Inc.,B1,F2 Primary products,No website text was supplied; the role of separately sold proprietary products is uncertain.
1,with_pages · e · default,Crawford & Company,B1,F2 Primary products,The pages mention a historical water-detection product launch but do not establish a current standalone product offe...


Companies the model split into more than one business:

In [15]:
split = test[test["business_count"] > 1]
split[["label", "target_ciq_ticker", "target_company_name", "business_id", "business"]] if len(split) else print("none split")

,label,target_ciq_ticker,target_company_name,business_id,business
1450,knowledge_only_i_haiku · i · claude-haiku-5-5,SBC,SBC Medical Group Holdings Incorporated,B1,aesthetic clinics
1451,knowledge_only_i_haiku · i · claude-haiku-5-5,SBC,SBC Medical Group Holdings Incorporated,B2,medical staffing
1452,knowledge_only_i_haiku · i · claude-haiku-5-5,ALB,Albemarle Corporation,B1,lithium chemicals
1453,knowledge_only_i_haiku · i · claude-haiku-5-5,ALB,Albemarle Corporation,B2,bromine specialties
1454,knowledge_only_i_haiku · i · claude-haiku-5-5,ALB,Albemarle Corporation,B3,refining and polymer catalysts
1455,knowledge_only_i_haiku · i · claude-haiku-5-5,JOE,The St. Joe Company,B1,real estate development and leasing
1456,knowledge_only_i_haiku · i · claude-haiku-5-5,JOE,The St. Joe Company,B2,hospitality and leisure
1457,knowledge_only_i_haiku · i · claude-haiku-5-5,JOE,The St. Joe Company,B3,forestry and timberlands
1460,knowledge_only_i_haiku · i · claude-haiku-5-5,AZZ,AZZ Inc.,B1,hot-dip galvanizing services
1461,knowledge_only_i_haiku · i · claude-haiku-5-5,AZZ,AZZ Inc.,B2,electrical and industrial products


**Agreement between runs** on the category facets, against the Codex fill at version f (largest business, B1, companies in both): the share of companies given the same categories. A facet counts only where both runs gave it a category (company size became a category at version h). Versions e and f differ in the prompt as well as the run, so the e column shows how much answers move even with the same model; a new model or prompt is judged against that, not against 100%.

In [16]:
REFERENCE = "knowledge_only · f · default"
same_set = lambda s: "; ".join(sorted(s.split("; ")))     # "B2B; B2G" == "B2G; B2B"
b1 = test[test["business_id"] == "B1"].set_index(["label", "IQ_COMPANY_ID"])
ref = b1.loc[REFERENCE]
agreement = {}
for label in b1.index.get_level_values("label").unique().drop(REFERENCE):
    other = b1.loc[label]
    common = ref.index.intersection(other.index)
    row = {}
    for f in fc.FACETS:
        if f.kind == "text":
            continue
        a, b = (t.loc[common, f"{f.name}_category"].fillna("") for t in (ref, other))
        both = a.ne("") & b.ne("")           # only where both runs gave a category
        row[f"{fc.FACET_ID[f.name]} {f.label}"] = 100 * (a[both].map(same_set) == b[both].map(same_set)).mean() \
            if both.any() else None
    agreement[f"{label} ({len(common)} companies)"] = row
pd.DataFrame(agreement).round(1) if agreement else print("only one run so far")

,knowledge_only · e · default (21 companies),knowledge_only_i_haiku · i · claude-haiku-5-5 (21 companies),with_pages · e · default (11 companies)
F4 Customer segment,90.5,71.4,81.8
F6 Revenue model,61.9,52.4,45.5
F7 Value chain position,76.2,42.9,81.8
F8 Geography served,90.5,90.5,100.0
F9 Company size,NaN,NaN,NaN
F10 Product specialization,100.0,71.4,90.9
F11 Distribution channels,85.7,28.6,63.6
F12 Regulatory exposure,85.7,81.0,90.9


For the runs at the current version, how their categories differ from the reference run (B1, companies with categories in both): the same set, a superset or a subset of the reference's categories, at least one in common, and how many categories per company. For a facet where several categories apply, a run that lists fewer or more of them is not the same as one that picks different ones.

In [17]:
rows = []
for label in test.loc[test["prompt_version"] == fc.PROMPT_VERSION, "label"].unique():
    other = b1.loc[label]
    common = ref.index.intersection(other.index)
    for f in fc.FACETS:
        if f.kind == "text":
            continue
        a, b = (t.loc[common, f"{f.name}_category"].fillna("") for t in (ref, other))
        both = a.ne("") & b.ne("")
        if not both.any():
            continue
        sa, sb = (x[both].map(lambda v: set(v.split("; "))) for x in (a, b))
        rows.append({"run": label, "facet": f"{fc.FACET_ID[f.name]} {f.label} ({f.kind})", "companies": int(both.sum()),
                     "same %": round(100 * (sa == sb).mean()), "more %": round(100 * (sb > sa).mean()),
                     "fewer %": round(100 * (sb < sa).mean()),
                     "one in common %": round(100 * sa.combine(sb, lambda x, y: bool(x & y)).mean()),
                     "categories per company": f"{sa.map(len).mean():.1f} -> {sb.map(len).mean():.1f}"})
pd.DataFrame(rows).set_index(["run", "facet"]) if rows else print("no run at the current version yet")

no run at the current version yet


**What the fill test at version i showed (2026-10-24):**
- **All 31 calls answered.** The model knew 29 companies and filled them in the first pass; AMETEK's primary services, the one facet it left unknown, came from the pages in the second pass.
- **The two it didn't know went to the pages, as designed.** Cardinal Infrastructure Group was filled entirely from its pages, every facet citing page lines that exist. Englewood Lab (a cosmetics contract maker listed in Korea, which Codex's larger model knew and Haiku doesn't) first stayed unknown: its saved "page" was the browser's certificate-error screen. The scraper now treats that screen as a failed load and tries the site's http:// address, and switches a homepage in another language (/ko/home) to its English copy; after a rescrape of that one company, the second pass filled all 12 facets from its English pages. A rescrape changes the second-pass prompt, so the second pass runs again on its own.
- **Five companies were split into businesses:** Albemarle and St. Joe into three, AZZ, Con Ed and SBC Medical into two. Ingles and Huron stayed one business.
- **Against the Codex answers (version f):** on the facets where several categories apply, Haiku mostly lists fewer of them and nearly always shares at least one, so the low "same set" shares there are about how many categories, not which. The single-choice facets differ by judgment (geography, regulatory exposure, specialization). Which answers are right is for the manual review, not this table.
- **Cost and time:** $0.0011 and about 9 seconds per company in the first pass; about $0.04 for the 31 with both passes, including the retries.